# ব্যয় দাবির বিশ্লেষণ

এই নোটবুকটি দেখায় কীভাবে এজেন্ট তৈরি করতে হয় যারা প্লাগইন ব্যবহার করে স্থানীয় রসিদ ইমেজ থেকে ভ্রমণ ব্যয় প্রক্রিয়া করে, একটি ব্যয় দাবি ইমেল তৈরি করে, এবং পাই চার্ট ব্যবহার করে ব্যয়ের তথ্য দৃশ্যায়ন করে। এজেন্টরা কাজের প্রসঙ্গ ভিত্তিতে ডায়নামিকভাবে ফাংশন নির্বাচন করে।

ধাপসমূহ:
1. ওসিআর এজেন্ট স্থানীয় রসিদ ইমেজ প্রক্রিয়া করে এবং ভ্রমণ ব্যয়ের তথ্য বের করে।
2. ইমেল এজেন্ট ব্যয় দাবির একটি ইমেল তৈরি করে।

### একটি ভ্রমণ ব্যয়ের দৃশ্যমান উদাহরণ:
ধরুন, আপনি একটি ব্যবসায়িক মিটিংয়ে অন্য শহরে যাত্রী। আপনার কোম্পানির নীতি সব যুক্তিযুক্ত ভ্রমণ সংক্রান্ত ব্যয় ফেরত দেওয়া। এখানে সম্ভাব্য ভ্রমণ ব্যয়ের বিবরণ:
- পরিবহন:
বাড়ি থেকে গন্তব্য শহরে যাওয়া এবং আসার জন্য এয়ারফেয়ার।
বিমানবন্দর যাওয়া এবং আসার জন্য ট্যাক্সি বা রাইড-হেলিং পরিষেবা।
গন্তব্য শহরে স্থানীয় পরিবহন (যেমন পাবলিক ট্রানজিট, রেন্টাল গাড়ি, বা ট্যাক্সি)।

- বাসস্থান:
মিটিং স্থলের কাছে মাঝারি মানের ব্যবসায়িক হোটেলে তিন রাত থাকার খরচ।

- ভোজ:
কোম্পানির প্রতিদিনের ভাতা নীতি অনুযায়ী নিয়মিত সকালের নাস্তা, দুপুরের খাবার, এবং রাতে খাবারের জন্য ভাতা।

- অন্যান্য ব্যয়:
বিমানবন্দরের পার্কিং ফি।
হোটেলে ইন্টারনেট ব্যবহারের চার্জ।
টিপস বা ছোট সার্ভিস চার্জ।

- দলিলপত্র:
আপনি সব রসিদ (ফ্লাইট, ট্যাক্সি, হোটেল, ভোজ, ইত্যাদি) এবং একটি পূর্ণাঙ্গ ব্যয় রিপোর্ট ফেরতপ্রাপ্তির জন্য জমা দেন।


## প্রয়োজনীয় লাইব্রেরি আমদানি করুন

নোটবুকটির জন্য প্রয়োজনীয় লাইব্রেরি এবং মডিউল গুলো আমদানি করুন।


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## খরচ মডেলগুলি সংজ্ঞায়িত করুন

 পৃথক খরচের জন্য একটি Pydantic মডেল এবং একটি ExpenseFormatter ক্লাস তৈরি করুন যা ব্যবহারকারীর প্রশ্নকে কাঠামোবদ্ধ খরচ ডাটাতে রূপান্তর করে।

 প্রতিটি খরচ নিম্নলিখিত বিন্যাসে উপস্থাপিত হবে:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## যন্ত্রপাতি সংজ্ঞায়িত করা - ইমেইল তৈরি করা

একটি টুল ফাংশন তৈরি করুন যা একটি খরচ দাবির জন্য ইমেইল তৈরি করে।
- এই টুলটি Microsoft Agent Framework থেকে `@tool` ডেকোরেটর ব্যবহার করে।
- এটি খরচের মোট পরিমাণ গণনা করে এবং বিস্তারিত ইমেইলের শরীরে ফরম্যাট করে।


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# রসিদ ছবিগুলো থেকে সফর খরচ বের করার জন্য টুল

রসিদ ছবিগুলো থেকে সফর খরচ বের করার জন্য একটি টুল ফাংশন তৈরি করুন।
- এই টুলটি Microsoft Agent Framework থেকে `@tool` ডেকোরেটর ব্যবহার করে।
- এটি রসিদ ছবিটি পড়ে, এটিকে base64 এ এনকোড করে, এবং এজেন্ট বিশ্লেষণের জন্য ডেটা URI রিটার্ন করে।


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## প্রসেসিং খরচ

এজেন্টগুলিকে সংজ্ঞায়িত করুন এবং তাদের একটি ধারাবাহিক ওয়ার্কফ্লোতে যুক্ত করুন `WorkflowBuilder` ব্যবহার করে।
- OCR এজেন্ট রসিদ ইমেজ থেকে কাঠামোবদ্ধ খরচের ডেটা বের করে `load_receipt_image` টুল ব্যবহার করে।
- ইমেইল এজেন্ট উত্তোলিত ডেটা নিয়ে একটি পেশাদারী খরচ দাবির ইমেইল তৈরি করে `generate_expense_email` টুল ব্যবহার করে।
- `WorkflowBuilder` এর `add_edge` দিয়ে একটি ধারাবাহিক পাইপলাইন তৈরি করে: OCR Agent → Email Agent।


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## প্রধান ফাংশন

ক্রমানুসারে কার্যপ্রবাহ তৈরি করুন এবং এটি চালান রসিদ ছবিটি প্রক্রিয়াকরণ করতে এবং ব্যয় দাবির ইমেল তৈরি করতে।


> **নোট:** এই ওয়ার্কফ্লোটি বর্তমানে রসিদের চিত্রটিকে বেস64 টেক্সট হিসেবে পাঠাচ্ছে, যেটিকে বেশিরভাগ চ্যাট মডেল (gpt-5-mini সহ) চিত্র হিসেবে গ্রহণ করবে না।
> এটি মডেলের কনটেক্সট উইন্ডোও অতিক্রম করতে পারে। Azure AI Vision (অথবা অন্য কোনো OCR টুল) দিয়ে OCR চালানো এবং কেবল নিষ্কাশিত টেক্সট পাঠানো পছন্দ করুন, অথবা চিত্রটি `image_url` বার্তা হিসেবে পাঠানোর জন্য পুনর্গঠন করুন।
> আপনি যদি কেবল কনটেক্সট ত্রুটি এড়াতে চান, তাহলে একটি ছোট রসিদ চিত্র বা বড় কনটেক্সট উইন্ডোসহ মডেল ব্যবহার করার চেষ্টা করুন।


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
